## Delta Parameter Modeling for Surgical Optimization
This notebook trains predictive machine learning models to estimate postoperative correction for key spinopelvic alignment parameters.

The trained models are saved and later used inside the optimization pipeline to simulate surgical plan outcomes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import PredictionErrorDisplay

from xgboost import XGBRegressor

from src import config

pd.set_option('display.max_columns', None)

## Data and Preprocessing

We load the cleaned and preprocessed dataset defined in config.DATA_PROCESSED.
This dataset contains preoperative alignment parameters, surgical plan variables, and postoperative outcomes.


In [ ]:
# Load cleaned data from config path

df = pd.read_csv(config.DATA_PROCESSED)
# df.columns = df.columns.astype(str).str.replace("\n"," ").str.replace(r"\s+"," ", regex=True).str.strip()

print(f"Loaded {df.shape[0]} patients, {df.shape[1]} columns")
print(f"Data path: {config.DATA_PROCESSED}")

## Modeling Approach

For each alignment parameter, a regression model is trained to predict the postoperative change, defined as the difference between postoperative and preoperative values. The models use patient demographics, baseline alignment measurements, and surgical plan variables as inputs. A shared preprocessing pipeline handles missing values and encodes categorical variables, with feature scaling applied where needed. Different regression algorithms are selected depending on the parameter to balance model flexibility and stability while keeping the overall framework consistent for integration into the optimization workflow.


In [ ]:
df

Some clinical variables (e.g., ODI scores) may be stored as strings.
We explicitly convert these to numeric types to ensure valid arithmetic operations when computing delta values.

In [ ]:
# Ensure ODI columns are numeric
df["ODI_preop"] = pd.to_numeric(df["ODI_preop"], errors="coerce")
df["ODI_12mo"]  = pd.to_numeric(df["ODI_12mo"], errors="coerce")

For each alignment parameter, the postoperative correction is computed as the difference between the postoperative and preoperative values. These delta variables serve as the regression targets for the modeling process. If a delta column is not already present in the dataset, it is created to ensure consistency across runs.

In [ ]:
# Ensure delta columns exist
if "delta_SVA" not in df.columns:
    df["delta_SVA"] = df["SVA_postop"] - df["SVA_preop"]

if "delta_SS" not in df.columns:
    df["delta_SS"] = df["SS_postop"] - df["SS_preop"]
    
# Ensure delta_GlobalTilt exists
if "delta_GlobalTilt" not in df.columns:
    df["delta_GlobalTilt"] = (
        df["global_tilt_postop"] - df["global_tilt_preop"]
    )
if "delta_LL" not in df.columns:
    df["delta_LL"] = df["LL_postop"] - df["LL_preop"]

if "delta_L1PA" not in df.columns:
    df["delta_L1PA"] = df["L1PA_postop"] - df["L1PA_preop"]

if "delta_T4PA" not in df.columns:
    df["delta_T4PA"] = df["T4PA_postop"] - df["T4PA_preop"]

if "delta_L4S1" not in df.columns:
    df["delta_L4S1"] = df["L4_S1_postop"] - df["L4S1_preop"]

if "delta_ODI" not in df.columns:
    df["delta_ODI"] = df["ODI_12mo"] - df["ODI_preop"]


Model inputs are constructed using patient fixed preoperative parameters and surgical plan variables defined in the shared configuration file. GAP-related predictors are excluded to avoid redundancy and potential leakage. The final feature list combines baseline patient characteristics with operative planning variables.

In [ ]:
# Use shared config for features

PATIENT_FIXED_COLS = config.PATIENT_FIXED_COLS
PLAN_COLS = config.PLAN_COLS

FEATURES = config.DELTA_MODEL_FEATURES.copy()

# Treat binary indicators as categorical
for col in config.BINARY_INDICATOR_COLS:
    if col in df.columns:
        df[col] = df[col].astype("category")

# Automatically split by dtype
NUMERIC_FEATURES = df[FEATURES].select_dtypes(
    exclude=["object", "string", "category"]
).columns.tolist()

CATEGORICAL_FEATURES = df[FEATURES].select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()


Features are automatically separated into numeric and categorical variables based on data type. Numeric features are imputed using the median, while categorical features are imputed and one-hot encoded. For linear models such as Ridge regression, numeric features are additionally standardized. These preprocessing steps are embedded within model pipelines to ensure consistency during training and deployment.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", TargetEncoder(smooth="auto"))
])

ridge_categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", TargetEncoder(smooth="auto")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES)
    ]
)
ridge_numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ridge_preprocessor = ColumnTransformer(
    transformers=[
        ("num", ridge_numeric_transformer, NUMERIC_FEATURES),
        ("cat", ridge_categorical_transformer, CATEGORICAL_FEATURES)
    ]
)

## Model Training

A unified training function is defined to ensure consistency across all delta models. The function splits the data into training and testing sets, applies the appropriate preprocessing pipeline, fits the specified regression model, and reports performance metrics including mean absolute error (MAE) and R². Ridge models use a scaled preprocessing pipeline, while tree-based models use the standard pipeline. This structure keeps model training consistent while allowing flexibility in algorithm choice.

In [ ]:
def train_model(df, target_col, model, model_name, use_ridge=False):

    from sklearn.base import clone

    data = df[FEATURES + [target_col]].dropna(subset=[target_col])

    X = data[FEATURES]
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    if use_ridge:
        base_preprocessor = clone(ridge_preprocessor)
    else:
        base_preprocessor = clone(preprocessor)

    base_preprocessor.fit(X_train, y_train)

    X_train_trans = base_preprocessor.transform(X_train)
    X_test_trans  = base_preprocessor.transform(X_test)

    if hasattr(X_train_trans, "toarray"):
        X_train_trans = X_train_trans.toarray()
        X_test_trans = X_test_trans.toarray()

    model_copy = clone(model)
    model_copy.fit(X_train_trans, y_train)

    preds = model_copy.predict(X_test_trans)

    print(f"\n{model_name}")
    print("MAE:", round(mean_absolute_error(y_test, preds), 3))
    print("R² :", round(r2_score(y_test, preds), 3))
    

    pipe = Pipeline([
        ("preprocess", base_preprocessor),
        ("model", model_copy)
    ])

    return pipe, X_train, X_test, y_train, y_test, preds

## Model Evaluation

Model performance is evaluated using mean absolute error (MAE) and R². These metrics summarize prediction accuracy and the proportion of variance explained by each model on a held-out test set. The evaluation step provides an initial assessment of predictive performance before training the final full-data models for deployment.


In [ ]:
# Define models
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=8,
    min_samples_leaf=5,
    oob_score=True
)

ridge_model = Ridge(alpha=1)

In [ ]:
sva_model, X_train_sva, X_test_sva, y_train_sva, y_test_sva, y_pred_sva = \
    train_model(df, "delta_SVA", xgb_model, "ΔSVA Model")

ss_model, X_train_ss, X_test_ss, y_train_ss, y_test_ss, y_pred_ss = \
    train_model(df, "delta_SS", xgb_model, "ΔSS Model")

gt_model, X_train_gt, X_test_gt, y_train_gt, y_test_gt, y_pred_gt = \
    train_model(df, "delta_GlobalTilt", xgb_model, "ΔGT Model")

ll_model, X_train_ll, X_test_ll, y_train_ll, y_test_ll, y_pred_ll = \
    train_model(df, "delta_LL", rf_model, "ΔLL Model")

l1pa_model, X_train_l1pa, X_test_l1pa, y_train_l1pa, y_test_l1pa, y_pred_l1pa = \
    train_model(df, "delta_L1PA", ridge_model, "ΔL1PA Model", use_ridge=True)

t4pa_model, X_train_t4pa, X_test_t4pa, y_train_t4pa, y_test_t4pa, y_pred_t4pa = \
    train_model(df, "delta_T4PA", ridge_model, "ΔT4PA Model", use_ridge=True)

l4s1_model, X_train_l4s1, X_test_l4s1, y_train_l4s1, y_test_l4s1, y_pred_l4s1 = \
    train_model(df, "delta_L4S1", ridge_model, "ΔL4S1 Model", use_ridge=True)

odi_model, X_train_odi, X_test_odi, y_train_odi, y_test_odi, y_pred_odi = \
    train_model(df, "delta_ODI", ridge_model, "ΔODI Model", use_ridge=True)

In [ ]:
def build_full_model(df, target_col, model, use_ridge=False):

    data = df[FEATURES + [target_col]].dropna(subset=[target_col])
    X = data[FEATURES]
    y = data[target_col]

    if use_ridge:
        pipe = Pipeline([
            ("preprocess", clone(ridge_preprocessor)),
            ("model", model)
        ])
    else:
        pipe = Pipeline([
            ("preprocess", clone(preprocessor)),
            ("model", model)
        ])

    pipe.fit(X, y)
    return pipe

In [ ]:
import joblib
from src import config

def save_model(pipe, target_col, model_name):

    target_dir = config.ARTIFACTS_DIR / target_col.replace("delta_", "")
    target_dir.mkdir(parents=True, exist_ok=True)

    bundle = {
        "pipe": pipe,
        "features": FEATURES,
        "target": target_col,
        "model_name": model_name
    }

    out_path = target_dir / f"{target_col}_model.joblib"
    joblib.dump(bundle, out_path)

    print("Saved:", out_path)

In [ ]:
save_model(sva_model,  "delta_SVA", "XGBRegressor")
save_model(ss_model,   "delta_SS", "XGBRegressor")
save_model(gt_model,   "delta_GlobalTilt", "XGBRegressor")

save_model(ll_model,   "delta_LL", "RandomForest")
save_model(l1pa_model, "delta_L1PA", "Ridge")
save_model(t4pa_model, "delta_T4PA", "Ridge")
save_model(l4s1_model, "delta_L4S1", "Ridge")
save_model(odi_model,  "delta_ODI", "Ridge")

## Model Diagnostics

For each alignment parameter, diagnostic plots are used to evaluate prediction behavior. These visualizations include predicted vs. actual values, residual distributions, residuals vs. predicted values, and absolute error distributions. Together, they provide insight into model fit, error spread, and potential bias patterns.

In [ ]:
def plot_diagnostics_3panel(y_test, y_pred, title):

    residuals = y_test - y_pred

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=14)

    # Actual vs Predicted
    axes[0].scatter(y_test, y_pred, alpha=0.6, marker="s")
    axes[0].plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        linestyle="--",
    )
    axes[0].set_title("Actual vs Predicted")
    axes[0].set_xlabel("Postoperative")
    axes[0].set_ylabel("Predicted")

    # Residual vs Predicted
    axes[1].scatter(y_pred, residuals, alpha=0.6)
    axes[1].axhline(0, linestyle="--")
    axes[1].set_title("Residuals vs Predicted")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Residual")

    # Residual Histogram
    sns.histplot(residuals, bins=15, kde=True, ax=axes[2])
    axes[2].set_title("Residual Histogram")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_shap_xgb(model, X_test, title):

    import shap
    import numpy as np

    preprocess = model.named_steps["preprocess"]
    final_model = model.named_steps["model"]

    X_test_trans = preprocess.transform(X_test)

    if hasattr(X_test_trans, "toarray"):
        X_test_trans = X_test_trans.toarray()

    feature_names = preprocess.get_feature_names_out()

    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test_trans)

    shap.summary_plot(
        shap_values,
        X_test_trans,
        feature_names=feature_names,
        max_display=15,
        show=False
    )
    plt.title(f"Top Feature Drivers of {model}")
    plt.show()
    

In [ ]:
def plot_shap_tree(model, X_train, X_test, title):

    import shap
    import numpy as np

    preprocess = model.named_steps["preprocess"]
    final_model = model.named_steps["model"]

    X_test_trans = preprocess.transform(X_test)

    if hasattr(X_test_trans, "toarray"):
        X_test_trans = X_test_trans.toarray()

    feature_names = preprocess.get_feature_names_out()

    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test_trans)

    shap.summary_plot(
        shap_values,
        X_test_trans,
        feature_names=feature_names,
        max_display=15,
        show=False
    )

    plt.title(f"Top Feature Drivers of {model}")
    plt.show()

In [ ]:
def plot_shap_linear(model, X_train, X_test, title):

    import shap
    import numpy as np

    preprocess = model.named_steps["preprocess"]
    final_model = model.named_steps["model"]

    # Transform
    X_train_trans = preprocess.transform(X_train)
    X_test_trans  = preprocess.transform(X_test)

    # Convert sparse to dense if needed
    if hasattr(X_train_trans, "toarray"):
        X_train_trans = X_train_trans.toarray()
    if hasattr(X_test_trans, "toarray"):
        X_test_trans = X_test_trans.toarray()

    feature_names = preprocess.get_feature_names_out()

    explainer = shap.LinearExplainer(final_model, X_train_trans)
    shap_values = explainer(X_test_trans)

    print(f"\n{title} — SHAP Beeswarm")

    shap.summary_plot(
        shap_values,
        X_test_trans,
        feature_names=feature_names,
        max_display=15
    )

In [ ]:
plot_diagnostics_3panel(
    y_test_sva,
    y_pred_sva,
    "ΔSVA Model Diagnostics"
)
plot_shap_xgb(
    sva_model,
    X_test_sva,
    "ΔSVA Model"
)

In [ ]:
plot_diagnostics_3panel(
    y_test_ss,
    y_pred_ss,
    "ΔSS Model Diagnostics"
)
plot_shap_xgb(
    ss_model,
    X_test_ss,
    "ΔSS Model"
)

In [ ]:
plot_diagnostics_3panel(
    y_test_gt,
    y_pred_gt,
    "ΔGlobal Tilt Model Diagnostics"
)
plot_shap_xgb(
    gt_model,
    X_test_gt,
    "ΔGT Model"
)

In [ ]:
plot_diagnostics_3panel(
    y_test_ll,
    y_pred_ll,
    "ΔLL Model Diagnostics"
)

plot_shap_tree(
    ll_model,
    X_train_ll,
    X_test_ll,
    "ΔLL Model"
)

In [ ]:
plot_diagnostics_3panel(
    y_test_l1pa,
    y_pred_l1pa,
    "ΔL1PA Model Diagnostics"
)

plot_shap_linear(
    l1pa_model,
    X_train_l1pa,
    X_test_l1pa,
    "ΔL1PA Model"
)

In [ ]:
plot_diagnostics_3panel(
    y_test_t4pa,
    y_pred_t4pa,
    "ΔT4PA Model Diagnostics"
)

plot_shap_linear(
    t4pa_model,
    X_train_t4pa,
    X_test_t4pa,
    "ΔT4PA Model"
)

In [ ]:
plot_diagnostics_3panel(
    y_test_l4s1,
    y_pred_l4s1,
    "ΔL4S1 Model Diagnostics"
)

plot_shap_linear(
    l4s1_model,
    X_train_l4s1,
    X_test_l4s1,
    "ΔL4S1 Model"
)

In [ ]:
plot_diagnostics_3panel(
    y_test_odi,
    y_pred_odi,
    "ΔODI Model Diagnostics"
)

plot_shap_linear(
    odi_model,
    X_train_odi,
    X_test_odi,
    "ΔODI Model"
)